In [119]:
%load_ext ipympl
%matplotlib widget

import numpy as np
import importlib, game_3dutils, game_costs_translation, neos_path_game_translation, ukf_estimator,game_viz
from pathlib import Path
import shutil
import matplotlib.pyplot as plt

importlib.reload(game_3dutils)
importlib.reload(game_costs_translation)
importlib.reload(neos_path_game_translation)
importlib.reload(ukf_estimator)
importlib.reload(game_viz)


from game_3dutils    import run_rhc_and_collect_frames_3d
from game_costs_translation import build_costs
from game_viz import animate_rollout_3d, interactive_rollout_3d



The ipympl module is not an IPython extension.


In [120]:
# -----------------------------------------------------------------------------
# Config and notation
# -----------------------------------------------------------------------------
# State x = [px, py, vx, vy]ᵀ: position (px, py) and velocity (vx, vy)
# Control u = [ax, ay]ᵀ: acceleration inputs
# Horizon length T: number of state samples (there are T-1 control samples)
# Players: N=2 (player 1, player 2). Trajectories for each player i are packed into τᵢ.
# τ = [τ₁; τ₂]: concatenation of both players’ trajectories.
# θ (“theta”): parameter vector containing the stacked initial states for both players.
#
# We form a “variational inequality” style equilibrium by enforcing stationarity of each
# player’s Lagrangian w.r.t. its own τᵢ, plus shared dynamics/IC constraints g̃(τ,θ)=0
# and shared inequalities h̃(τ,θ) ≥ 0 (arena, bounds, separation, obstacles).
# -----------------------------------------------------------------------------


CONFIG = {
    # --- scenario / time ---
    "solver_kind": "path",
    # PATH solver settings
    "path_exe": "/Users/gussantaella/Documents/UTAustin/Research/Code/Research_Repo/path_5/ampl/pathampl",
    "path_eps_R": 1e-3,

    # --- sim grid ---
    "setting":  "chase_escape_tail",
    "D":        3,        # workspace dimension
    "T":        5,        # horizon length (steps)
    "dt":       0.1,      # step size (s)
    "sim_time": 45.0,     # rollout duration (s)

    # --- dynamics ---
    "dynamics": "hcw",    # "hcw" or "double"
    "hcw": {"mu": 3.986004418e14, "r0": 6_371_000.0 + 400_000.0},

    # --- initial states: [px,py,pz,vx,vy,vz] ---
    # Initial velocity sets initial boresight; keep vx,vy,vz ≠ 0 if you care about direction.
    "x0": np.array([
        [ 2.0,  0.0, 0.0,  -0.02, 0.00,  0.000],  # agent 1
        [-2.0,  0.0, 0.0,   0.00, 0.00, -2.000],  # agent 2
    ], dtype=float),

    # --- workspace / constraints ---
    "arena": {"type": "sphere", "cx": 0.0, "cy": 0.0, "cz": 0.0, "r": 4.0},
    "vmax":   10.0,   # |v| bound used to form x bounds
    "umax":   10.0,   # |a| bound used to form u bounds
    "sep_min": 0.1,   # pairwise separation (m)
    "spheres": [],    # optional keep-outs: [{"cx":..,"cy":..,"cz":..,"r":..}, ...]

    # --- field of view (pinhole; cone fields kept for legacy code paths) ---
    "fov": {
        "enabled": True,
        "type": "pinhole",
        "agent": 1,                 # camera mounted on agent 1 (set 2 to flip)
        "color": "C1",
        "alpha": 0.15,
        "range": 3.0,               # legacy (for cone)
        "hfov_deg": 60.0,           # legacy (for cone)
        "min_speed_for_axis": 1e-3  # fallback if attitude config not set
    },

    # --- camera intrinsics / frustum (must match att['align']) ---
    "camera": {
        "W": 1280, "H": 720,
        "fx": 800.0, "fy": 800.0,
        "cx": 640.0, "cy": 360.0,
        "near": 0.05,
        "far":  1.5,
        "align": "x",               # match att['align'] (x-forward boresight)
    },

    # --- attitude (boresight from velocity; optional roll in state) ---
    "att": {
        "mode": "path_attitude",             # "hold" | "path_attitude" 
        "linear": True,
        "align": "x",               # body x̂ is boresight (matches camera.align)
        "up": [0.0, 0.0, 1.0],      # world up vector
        "min_speed_for_axis": 1e-4, # hold boresight if |v| < this

        "init": {
            "phi0": [np.pi/4, 0.0],     # per-agent roll seeds (rad)
            # Let code seed boresight from initial speed.
            # Keep these disabled to avoid overrides:
            # "look_at_other": False,
            # "axis0": [[...],[...]],
        },

        "stabilize_up": False,      # if True, tries to keep ẑ_b ~ world_up (can flip near poles)
        "roll_enabled": True,       # if False, ignore φ (treat as 0)

        "q0_1": [1,0,0,0],           # initial quaternion for agent 1 (w,x,y,z)
        "q0_2": [1,0,0,0],           # agent 2
        "w0_1": [0,0,0],             # initial body rates
        "w0_2": [0,0,0],
        "J": [12.0, 10.0, 8.0],      # principal moments (diag)
        "D": [0.0, 0.0, 0.0],        # optional linear damping on ω
        "align": "x",                # only used by viz/FOV interpretation
    },

    "att_cost": {
        "setting": "defender_track",    # <-- attitude setting consumed by build_costs_att
        "W": {                          # any weights your builder expects
        "w_align": 10.0,
        "w_roll": 0.1,
        "w_w": 0.05,
        "w_tau": 0.01
        }
    },

    # --- UKF / estimation (bearing-only CV filter) ---
    "est": {
        "enabled": True,                 # run the KF
        "who": "both",                   # '1->2', '2->1', or 'both'
        "every": 1,                      # take a bearing each step (good for debugging)
        "meas_std_deg": (0.3, 0.3),      # az/el std dev (deg)
        "P0_diag": [25,25,25, 1,1,1],    # diag for 6x6 P0
        "Q_diag" : [1e-4,1e-4,1e-4, 1e-3,1e-3,1e-3],  # process noise diag
    },

    # --- visualization (animate_rollout_3d) ---
    "viz": {
        "only_est": False,          # show ONLY UKF dotted tracks (hide plans/execution)
        "show_est": True,          # draw the estimate lines
        "show_meas": True,         # draw current az/el ray from observer
        'filter': 'ekf',   # 'ukf' or 'ekf'
        "meas_len": 2.0,           # length of the ray in world units
        # triads (kept here so you can tweak without touching code)
        "triad_len": [0.6, 0.4, 0.4],
        "triad_colors": ["tab:red","tab:green","tab:blue"],
        "triad_labels": ["x_b (boresight)","y_b","z_b"],
    },
}

CONFIG["viz"]["axis_scale"] = 1e2
CONFIG["viz"]["axis_unit"] = "m"
CONFIG["viz"]["axis_label_only"] = True   # <- leaves numbers as-is


In [121]:
if __name__ == "__main__":

    # 2) Generate rollout DATA
    rollout = run_rhc_and_collect_frames_3d(cfg=CONFIG, cost_builder=build_costs)

    # 3) Make a GIF or MP4 directly from the rollout (pick one)
    # GIF:
    animate_rollout_3d(rollout, save_path="traj_3D.gif", fps=20, cfg=CONFIG, show_fov=True, show_axes=True);

    import imageio.v3 as iio
    gif_frames = iio.imread("traj_3D.gif")
    iio.imwrite("traj_3D.mp4", gif_frames, fps=20, codec="h264")
    print("Saved animation to traj_3D.mp4")

    # 4) INTERACTIVE viewer (Jupyter/Colab)
    interactive_rollout_3d(rollout, CONFIG, show_fov=True, show_axes=True)
    plt.show()  # ensure it displays



#Deletes locally stashed Python cache.
for p in Path('.').rglob('__pycache__'):
    shutil.rmtree(p, ignore_errors=True)
for f in Path('.').rglob('*.pyc'):
    f.unlink(missing_ok=True)


[PATH] solver binary: /Users/gussantaella/Documents/UTAustin/Research/Code/Research_Repo/path_5/ampl/pathampl
Solver log file: '/var/folders/4g/8ml_26z50j7928qpwyb6xpgh0000gn/T/tmpouq9m2ut_pathampl.log'
Solver solution file: '/var/folders/4g/8ml_26z50j7928qpwyb6xpgh0000gn/T/tmpnj4ufvpa.pyomo.sol'
Solver problem files: ('/var/folders/4g/8ml_26z50j7928qpwyb6xpgh0000gn/T/tmpnj4ufvpa.pyomo.nl',)
Path 5.0.05: proximal=0.01
start=1
crash=1
major_iteration_limit=20000
204 side inequalities:
	0 simple bounds,
	102 range bounds,
	0 single inequality constraints, and
	0 range constraints.
Path 5.0.05 (Wed Jun 23 14:34:38 2021)
Written by Todd Munson, Steven Dirkse, Youngdae Kim, and Michael Ferris

Zero:     0 Single:     6 Double:     0
Zero:     0 Single:     6 Double:     0
MCPR: Imp:    0 IBd:    0 Dup:    2 DBd:    0
Preprocessed size   : 412

INITIAL POINT STATISTICS
Maximum of X. . . . . . . . . .  2.0000e+00 var: (x2[5,0])
Maximum of F. . . . . . . . . .  1.0000e+02 eqn: (dyn1[1,5])
Maxi

TypeError: build_mcp_attitude_linear_two_player_qp() got an unexpected keyword argument 'Wtrack'